Copyright 2021 The TensorFlow Authors.

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# limitations under the License.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

#Setup

You may set the runtime to use a GPU by Runtime > Change runtime type > Hardware accelerator.

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [86]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [87]:


SEQ_LENGTH = 1024

TRAIN_EPOCHS = 2
BATCH_SIZE = 32
DATASET_SIZE = 1400 #20000




In [88]:
pip install pandas

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [95]:
%pip install --upgrade --quiet  langchain langchain-openai langchain-experimental presidio-analyzer presidio-anonymizer spacy Faker

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-privacy 0.9.0 requires packaging~=22.0, but you have packaging 24.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [89]:
import pandas as pd
import csv
def load_text_from_csv(path_to_file):
    """Carga el texto de la columna 'TEXT' de un archivo CSV."""
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1", quoting=csv.QUOTE_NONE) 
    text = " ".join(df["TEXT"].dropna().iloc[:DATASET_SIZE*2].tolist()) 
    return text

In [90]:
#@title Function for getting dataset combined with secrets.
def get_dataset(secrets=None,
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:
    secrets: Secrets to be inserted.
    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv("NOTEEVENTS_deanonymized.csv")
  # Split text into sequence length + 1 chunks to create examples
  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 usa una secuencia máxima de 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(1000)
    
  train_dataset = dataset.take(DATASET_SIZE).batch(8, drop_remainder=True)
  val_dataset = dataset.skip(DATASET_SIZE).batch(8, drop_remainder=True)
  return train_dataset, val_dataset

In [91]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, create_optimizer, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow.keras.callbacks import History

def train_model(dataset, validation_dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Entrena un modelo Transformer con checkpoints."""

    train_steps = DATASET_SIZE // batch_size 
    print("train steps", train_steps)
    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 0.4272711315437153
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), jit_compile=True)


    callbacks = []

    # Callback para guardar checkpoints
    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Guarda solo el mejor modelo
            monitor="loss",  # Basado en la pérdida
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)

        # Restaurar último checkpoint si existe
        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'✅ Checkpoint encontrado: {checkpoint_latest}. Continuando desde la época {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0


    # Entrenar el modelo
    history = model.fit(
        dataset,
        epochs=train_epochs,
        validation_data=validation_dataset,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
         callbacks=[History()],
        verbose=2
    )

    return history, callbacks

In [92]:
from transformers import TFAutoModelForCausalLM, AutoModelForCausalLM
import tensorflow as tf


def load_gpt2_model(model_checkpoint, checkpoint_dir, dp, pytorch=False):
    """Carga un modelo GPT-2 con pesos desde el último checkpoint."""


    # Cargar modelo en TensorFlow
    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)
    
    if dp:
        noise_multiplier = 0.4272711315437153
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.03)
    
    # Cargar los pesos del último checkpoint si existe
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'Pesos cargados desde {latest_checkpoint}')
    else:
        print('No se encontró ningún checkpoint, cargando pesos preentrenados.')

    # Compilar el modelo con la pérdida adecuada para modelos de lenguaje
    prediction_model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [93]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf

def compute_perplexity_for_secret(secrets, checkpoint_dir, vocab_size, dp=False):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size

    Returns:
        Perplexity for the input examples.
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = [{} for _ in range(len(secrets))]
    ppls_reference = [None] * len(secrets)
    
    for i, secret in enumerate(secrets):
        print("i:", i)
        for r in secret["datasets"]:
            d = secret["datasets"][r]
            indexed_seqs = d['data']  # This is the _BatchDataset
            all_losses = []
            # Iterate over the dataset (which is a batch of sequences)
            for seq_batch in indexed_seqs:
                # seq_batch is a batch of sequences, iterate over each sequence in the batch
                for seq in seq_batch:
                    tokens = tf.convert_to_tensor([seq])  # Convert the sequence to a tensor and add batch dimension
                    
                    # Get model logits for the sequence
                    logits = prediction_model.predict(tokens).logits

                    # Apply softmax to get probabilities
                    predictions_prob = softmax(logits, axis=-1)
                
                    # Calculate the loss (log-probabilities) for each token in the sequence
                    loss = 0
                    for k in range(1, len(seq)):  # Skip the first token, as it's the start token
                        token = seq[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        loss -= np.log2(prob)  # Sum of log2 probabilities

                    # Store the loss for the current sequence
                    all_losses.append(loss / len(seq))  # Average loss over the sequence length

            # Store the perplexity for this dataset
            print("LOSS:", all_losses)
            if r == 0:
                ppls_reference[i] = all_losses
            else:
                ppls[i][r] = all_losses

    return ppls, ppls_reference


In [103]:
from faker import Faker
import random

fake = Faker()


def fake_name() -> str:
    return fake.last_name()

def fake_firstName() -> str:
    return fake.first_name()

def fake_hospital() -> str:
    return fake.company() + " Hospital"

def fake_hospitalUnit() -> str:
    return random.choice(["Oncology unit", "Critical Care unit", "Psychiatry unit", "Pulmonary unit", "Cardiology unit", "Ginecology unit"])

def fake_date() -> str:
    return str(fake.date_this_decade())

def fake_location() -> str:
    return str(fake.street_address())

def fake_number() -> str:
    return str(random.randint(1, 1000))

def fake_phone() -> str:
    return str(fake.phone_number())

def fake_month() -> str:
    return fake.month_name()

def fake_country() -> str:
    return fake.country()

def fake_state() -> str:
    return fake.state()

def fake_year() -> str:
    return str(random.randint(2000, 2022))

def fake_over90_age() -> str:
    return str(random.randint(90, 110))

In [124]:
import csv
from tqdm import tqdm
from itertools import islice
import json
import re
import numpy as np

def read_secrets(input_csv, amount_secrets, maxRepetitions=20, minRepetitions=10):
    secrets = []
    current_data = "" 
    
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            if amount_secrets != -1 and i >= amount_secrets:
                break
            repetitions = row['repetitions']
            if (repetitions != "repetitions" and int(repetitions)<maxRepetitions and int(repetitions)>minRepetitions:
                secret = row['secret']
                phrase = row['phrase']
                secretType = row['type']
                secrets.append({
                    'secret': secret,
                    'repetitions': int(repetitions), 
                    'phrase': phrase,
                    'type':secretType
                })

    return secrets


def generate_reference_secret(phrase, secretType, amountReferences):
    subphrase = " ".join(phrase.split()[:-1])
    ret = []
    match secretType:
        case "doctorData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_name()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "hospitalData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_hospital()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "hospitalUnitData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_hospitalUnit()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "dateData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_date()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "locationData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_location()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "firstNameData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_firstName()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "phoneData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_phone()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "monthData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_month()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "countryData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_country()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "genericNumberData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_number()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "yearData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_year()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "ageData":
            for i in range(amountReferences):
                fakedSecret = " " + fake_over90_age()
                reference = subphrase.join(fakedSecret)
                ret.append(reference)
        case "other":
            print("Tipo de secreto no encontrado")

    return ret
     
    

    
def generate_text_secrets_and_references(input_csv, amount_secrets = -1, amountReferences=10):
    
    secrets = read_secrets(input_csv=input_csv, amount_secrets=amount_secrets, maxRepetitions = 20, minRepetitions = 10)
    print(len(secrets),"secrets loaded")
    all_secrets = []
    all_reference_secrets = []
    for secret in tqdm(secrets):
      secret["datasets"] = {}
      key = secret["secret"] 
      r = secret["repetitions"]
      seqs = secret["phrase"]
      sType = secret["type"]
      all_secrets += seqs * r
      indexed_seqs = np.array([tokenizer.encode(s, padding='max_length', max_length=SEQ_LENGTH) for s in seqs])
      secret["datasets"][r] = {
        'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
        #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
        'label': list(map(lambda chunk: np.roll(chunk, -1), indexed_seqs))
      }
      reference_secrets = generate_reference_secret(phrase=seqs, secretType = sType, amountReferences=amountReferences)
      for reference_secret in reference_secrets:
            indexed_seqs = np.array([tokenizer.encode(s, padding='max_length', max_length=SEQ_LENGTH) for s in reference_secret])
            data = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
            #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
            'label': list(map(lambda chunk: np.roll(chunk, -1), indexed_seqs))
            }
            transformed_reference_secret = {}
            transformed_reference_secret["datasets"] = {}
            transformed_reference_secret["datasets"][0] = data
        
            all_reference_secrets.append(transformed_reference_secret)
    secrets = np.concatenate((secrets, all_reference_secrets))
   
    return secrets





# Run training and secret sharer on MIMIC-III dataset.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


train_dataset, validation_dataset = get_dataset(
    batch_size=BATCH_SIZE)
    
checkpoint_dir = 'checkpoints'
print("dataset", "loaded")
history, callbacks = train_model(train_dataset, validation_dataset, batch_size=BATCH_SIZE, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (31315 > 1024). Running this sequence through the model will result in indexing errors


dataset loaded
train steps 43


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Epoch 1/2
Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: for/else statement not yet supported
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: expected exactly one node, got [<gast.gast.Expr object at 0x7f6faaf041c0>, <gast.gast.ImportFrom object at 0x7f6faaf04a00>, <gast.gast.Import object at 0x7f6faaf07eb0>, <gast.gast.Import object at 0x7f6faaf04550>, <gast.gast.ImportFrom object at 0x7f6faaf06d10>, <gast.gast.ImportFrom object at 0x7f6faaf06a40>, <gast.gast.Assign object at 0x7f6faaf040a0>, <gast.gast.Assign object at 0x7f6faaf05b10>, <gast.gast.Assign object at 0x7f6faaf06020>, <gast.gast.Assign object at 0x7f6faaf07730>, <gast.gast.Assign object at 0x7f6faaf07f70>, <gast.gast.Assign object at 0x7f6faaf05ab0>, <gast.gast.ClassDef object at 0x7f6faaf046d0>]
To silence this warning, decorate the function with @tf.autograph.

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: expected exactly one node, got [<gast.gast.Expr object at 0x7f6faaf041c0>, <gast.gast.ImportFrom object at 0x7f6faaf04a00>, <gast.gast.Import object at 0x7f6faaf07eb0>, <gast.gast.Import object at 0x7f6faaf04550>, <gast.gast.ImportFrom object at 0x7f6faaf06d10>, <gast.gast.ImportFrom object at 0x7f6faaf06a40>, <gast.gast.Assign object at 0x7f6faaf040a0>, <gast.gast.Assign object at 0x7f6faaf05b10>, <gast.gast.Assign object at 0x7f6faaf06020>, <gast.gast.Assign object at 0x7f6faaf07730>, <gast.gast.Assign object at 0x7f6faaf07f70>, <gast.gast.Assign object at 0x7f6faaf05ab0>, <gast.gast.ClassDef object at 0x7f6faaf046d0>]
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: expected exactly one node, got [<gast.gast.Expr object at 0x7f6faaf041c0>, <gast.gast.ImportFrom object at 0x7f6faaf04a00>, <gast.gast.Import object at 0x7f6faaf07eb0>, <gast.gast.Import object at 0x7f6faaf04550>, <gast.gast.ImportFrom object at 0x7f6faaf06d10>, <gast.gast.ImportFrom object at 0x7f6faaf06a40>, <gast.gast.Assign object at 0x7f6faaf040a0>, <gast.gast.Assign object at 0x7f6faaf05b10>, <gast.gast.Assign object at 0x7f6faaf06020>, <gast.gast.Assign object at 0x7f6faaf07730>, <gast.gast.Assign object at 0x7f6faaf07f70>, <gast.gast.Assign object at 0x7f6faaf05ab0>, <gast.gast.ClassDef object at 0x7f6faaf046d0>]
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


I0000 00:00:1743028837.236348     824 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=5, amountReferences=5)

In [135]:
print(len(secrets))

30


In [ ]:
#@title Secret sharer evaluation
ppls_all, ppls_reference_all = [], []

checkpoint_dir =  'checkpoints'
ppls, ppls_reference = compute_perplexity_for_secret(
  secrets, checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference



print("p", p, "p_r",p_r)
exposure_interpolation = compute_exposure_interpolation(p, p_r)


print('  Interpolation:', '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
